In [ ]:
import duckdb
con = duckdb.connect("lol_rankings.duckdb")
con.execute(open("docs/schema.sql").read())   # creates the clean tables if missing

In [ ]:
con.execute("DELETE FROM team_aliases")   # delete children before parents
con.execute("DELETE FROM teams")
con.execute("""
    INSERT INTO teams (team_id, canonical_name, short, organization, region, is_disbanded, renamed_to)
    SELECT OverviewPage, COALESCE(Name, OverviewPage), Short, OrganizationPage, Region, IsDisbanded, RenamedTo
    FROM raw.teams
""")
print(con.execute("SELECT COUNT(*) FROM teams").fetchone())   # (1177,)

In [ ]:
con.execute("""
    INSERT INTO team_aliases (alias, team_id, change_type)
    SELECT lower(team_id), team_id, 'page' FROM teams
""")
con.execute("""
    INSERT INTO team_aliases (alias, team_id, change_type)
    SELECT lower(AllName), any_value(_pageName), 'redirect'
    FROM raw.team_redirects
    WHERE _pageName IN (SELECT team_id FROM teams)
      AND lower(AllName) NOT IN (SELECT alias FROM team_aliases)
    GROUP BY lower(AllName)
""")
print(con.execute("SELECT change_type, COUNT(*) FROM team_aliases GROUP BY 1").fetchall())

In [ ]:
print(con.execute("""
    SELECT t FROM (SELECT Team1 AS t FROM raw.match_schedule UNION SELECT Team2 FROM raw.match_schedule
                   UNION SELECT Blue FROM raw.match_schedule_game UNION SELECT Red FROM raw.match_schedule_game)
    WHERE lower(t) NOT IN (SELECT alias FROM team_aliases)
""").fetchall())   # []

In [ ]:
con.execute("DELETE FROM tournaments")   # child of region_map
con.execute("DELETE FROM region_map")
con.execute("""
    INSERT INTO region_map (league, year, region, league_level)
    SELECT t.League, CAST(t.Year AS INTEGER),
           CASE WHEN bool_or(t.Region = 'International') THEN 'International'
                ELSE any_value(t.Region) END,
           any_value(l.Level)
    FROM raw.tournaments t
    LEFT JOIN raw.leagues l ON l.League = t.League
    GROUP BY t.League, t.Year
""")
print(con.execute("SELECT * FROM region_map ORDER BY year, league").fetchall())

In [ ]:
con.execute("""
    INSERT INTO tournaments (tournament_id, overview_page, league, year, split, level, is_playoffs)
    SELECT OverviewPage, OverviewPage, League, CAST(Year AS INTEGER), Split, TournamentLevel, IsPlayoffs
    FROM raw.tournaments
""")
print(con.execute("SELECT COUNT(*), COUNT(*) FILTER (WHERE is_playoffs) FROM tournaments").fetchone())   # (78, 31)

In [ ]:
print(con.execute("""
    SELECT DISTINCT OverviewPage FROM raw.scoreboard_games
    WHERE OverviewPage NOT IN (SELECT tournament_id FROM tournaments)
""").fetchall())   # []

In [ ]:
con.execute("DELETE FROM matches")
con.execute("""
    INSERT INTO matches (match_id, tournament_id, datetime_utc, team1_id, team2_id, best_of, stage,
                         is_tiebreaker, is_forfeit, is_nullified, winner_id)
    SELECT m.MatchId, m.OverviewPage, m.DateTime_UTC, a1.team_id, a2.team_id, m.BestOf,
           COALESCE(m.Round, m.Tab), m.IsTiebreaker, COALESCE(m.FF, 0) > 0, m.IsNullified,
           CASE m.Winner WHEN '1' THEN a1.team_id WHEN '2' THEN a2.team_id END
    FROM raw.match_schedule m
    LEFT JOIN team_aliases a1 ON a1.alias = lower(m.Team1)
    LEFT JOIN team_aliases a2 ON a2.alias = lower(m.Team2)
""")
print(con.execute("SELECT COUNT(*), COUNT(team1_id), COUNT(team2_id), COUNT(winner_id) FROM matches").fetchone())

In [ ]:
con.execute("DELETE FROM games")
con.execute("""
    INSERT INTO games (game_id, match_id, riot_platform_game_id, game_num, datetime_utc, patch,
                       blue_team_id, red_team_id, winner_id, side_selection_team_id, first_pick_team_id,
                       is_remake, length_s)
    SELECT g.GameId, g.MatchId, g.RiotPlatformGameId, g.N_GameInMatch,
           COALESCE(s.DateTime_UTC, m.datetime_utc), COALESCE(s.Patch, ms.Patch),
           ab.team_id, ar.team_id,
           CASE g.Winner WHEN 1 THEN ab.team_id WHEN 2 THEN ar.team_id END,
           asel.team_id, afp.team_id,
           COALESCE(g.IsRemake, false), CAST(round(s.Gamelength_Number * 60) AS INTEGER)
    FROM raw.match_schedule_game g
    JOIN matches m ON m.match_id = g.MatchId
    JOIN raw.match_schedule ms ON ms.MatchId = g.MatchId
    LEFT JOIN raw.scoreboard_games s ON s.GameId = g.GameId
    LEFT JOIN team_aliases ab ON ab.alias = lower(g.Blue)
    LEFT JOIN team_aliases ar ON ar.alias = lower(g.Red)
    LEFT JOIN team_aliases asel ON asel.alias = lower(g.Selection)
    LEFT JOIN team_aliases afp ON afp.alias = lower(g.FirstPick)
    WHERE g.Winner IS NOT NULL
""")
print(con.execute("""
    SELECT COUNT(*), COUNT(blue_team_id), COUNT(red_team_id), COUNT(winner_id), COUNT(length_s)
    FROM games
""").fetchone())   # (5177, 5177, 5177, 5177, 5175)

In [ ]:
# winner agrees with the scoreboard
print(con.execute("""
    SELECT COUNT(*) FROM games g
    JOIN raw.scoreboard_games s ON s.GameId = g.game_id
    JOIN team_aliases a ON a.alias = lower(s.WinTeam)
    WHERE a.team_id <> g.winner_id
""").fetchone())   # (0,)

# each series has as many games as its score says
print(con.execute("""
    SELECT COUNT(*) FROM raw.match_schedule m
    WHERE m.Winner IS NOT NULL
      AND m.Team1Score + m.Team2Score <> (SELECT COUNT(*) FROM games g WHERE g.match_id = m.MatchId)
""").fetchone())   # (0,)

In [ ]:
con.close()